# Tugas 6 — Latihan Mandiri Pertemuan 9: Classical Problems
**Mata Kuliah:** Pemrosesan Paralel dengan Python

## Latihan 1 — Dining Philosophers dengan N Berbeda

Solusi arbitrator (`Semaphore(N-1)`) dari bagian 1 dijalankan ulang untuk **N = 3, 7, dan 10**. Setiap filsuf makan 20 kali; throughput = total makan ÷ waktu total (makan/detik). Kolom tambahan `maks. makan serentak` = ⌊N/2⌋, karena setiap filsuf memakai 2 garpu dari N garpu.

In [1]:
import threading
import time

def jalankan_dining(N, putaran=20):
    forks = [threading.Lock() for _ in range(N)]
    kursi_meja = threading.Semaphore(N - 1)   # arbitrator
    jumlah_makan = [0] * N

    def filsuf(i):
        kiri = forks[i]
        kanan = forks[(i + 1) % N]
        for _ in range(putaran):
            with kursi_meja:
                kiri.acquire()
                kanan.acquire()
                jumlah_makan[i] += 1
                time.sleep(0.02)      # makan
                kanan.release()
                kiri.release()
            time.sleep(0.01)          # berpikir

    threads = [threading.Thread(target=filsuf, args=(i,)) for i in range(N)]
    mulai = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    durasi = time.perf_counter() - mulai
    total = sum(jumlah_makan)
    return total, durasi, total / durasi

print(f"{'N':>3} | {'total makan':>11} | {'waktu (s)':>9} | {'throughput (makan/s)':>20} | {'maks makan serentak':>19} | {'throughput/N':>12}")
print("-" * 90)
hasil_throughput = {}
for N in (3, 7, 10):
    total, durasi, tp = jalankan_dining(N)
    hasil_throughput[N] = tp
    print(f"{N:>3} | {total:>11} | {durasi:>9.2f} | {tp:>20.1f} | {N//2:>19} | {tp/N:>12.1f}")

  N | total makan | waktu (s) | throughput (makan/s) | maks makan serentak | throughput/N
------------------------------------------------------------------------------------------


  3 |          60 |      1.22 |                 49.1 |                   1 |         16.4


  7 |         140 |      0.96 |                145.8 |                   3 |         20.8


 10 |         200 |      0.82 |                244.5 |                   5 |         24.4


**Jawaban: Throughput TIDAK naik linear (proporsional) terhadap N.**

Alasan:
1. Setiap filsuf butuh **2 garpu**, sedangkan garpu hanya ada **N**. Jadi jumlah filsuf yang bisa makan **bersamaan** dibatasi **⌊N/2⌋**, bukan N. Throughput mengikuti ⌊N/2⌋ (N=3 → 1, N=7 → 3, N=10 → 5), sehingga rasio throughput antar-N mengikuti rasio ⌊N/2⌋, bukan rasio N. Terlihat dari kolom `throughput/N` yang tidak konstan.
2. Karena pembulatan ke bawah, N ganjil "membuang" satu filsuf: N=3 hanya 1 yang makan serentak (bukan 1,5) dan N=7 hanya 3 (bukan 3,5).
3. Arbitrator `Semaphore(N-1)` juga membatasi jumlah filsuf yang berebut garpu, tetapi batas ini bukan penghambat utama; batas utamanya tetap jumlah garpu.

Jadi throughput naik seiring N, tetapi berbentuk tangga mengikuti ⌊N/2⌋ (kira-kira N/2 kali kecepatan satu filsuf), bukan linear terhadap N.

## Latihan 2 — Readers-Writers dengan Prioritas Writer

Solusi memakai `threading.Condition` dengan tiga status: `reader_aktif`, `writer_aktif`, dan `writer_menunggu`. Aturan prioritas writer: reader baru **menunggu selama masih ada writer yang menunggu atau sedang menulis**. Reader yang sudah membaca tetap boleh menyelesaikan bacaannya.

In [2]:
import threading
import time

class ReadWriteLockPrioritasWriter:
    def __init__(self):
        self.cond = threading.Condition()
        self.reader_aktif = 0
        self.writer_aktif = False
        self.writer_menunggu = 0

    def mulai_baca(self):
        with self.cond:
            # reader baru DITAHAN selama ada writer aktif ATAU writer yang menunggu
            while self.writer_aktif or self.writer_menunggu > 0:
                self.cond.wait()
            self.reader_aktif += 1

    def selesai_baca(self):
        with self.cond:
            self.reader_aktif -= 1
            if self.reader_aktif == 0:
                self.cond.notify_all()

    def mulai_tulis(self):
        with self.cond:
            self.writer_menunggu += 1
            while self.writer_aktif or self.reader_aktif > 0:
                self.cond.wait()
            self.writer_menunggu -= 1
            self.writer_aktif = True

    def selesai_tulis(self):
        with self.cond:
            self.writer_aktif = False
            self.cond.notify_all()

### Uji 1 — Prioritas writer berjalan benar
3 reader mulai membaca, lalu 2 writer datang, lalu 3 reader baru datang. Yang diharapkan: reader lama (0–2) selesai membaca, **kedua writer menulis lebih dulu**, baru reader baru (3–5) membaca; tidak pernah ada reader dan writer aktif bersamaan.

In [3]:
rw = ReadWriteLockPrioritasWriter()
data = {"nilai": 0}
log = []
log_lock = threading.Lock()
pelanggaran = [0]

def catat(m):
    with log_lock:
        log.append(m)

def reader(i):
    rw.mulai_baca()
    if rw.writer_aktif:
        pelanggaran[0] += 1
    catat(f"Reader-{i} mulai membaca nilai={data['nilai']}")
    time.sleep(0.05)
    rw.selesai_baca()

def writer(i):
    rw.mulai_tulis()
    if rw.reader_aktif > 0:
        pelanggaran[0] += 1
    data["nilai"] += 1
    catat(f"Writer-{i} menulis nilai baru={data['nilai']}")
    time.sleep(0.05)
    rw.selesai_tulis()

grup1 = [threading.Thread(target=reader, args=(i,)) for i in range(3)]
for t in grup1: t.start()
time.sleep(0.01)
grup2 = [threading.Thread(target=writer, args=(i,)) for i in range(2)]
for t in grup2: t.start()
time.sleep(0.01)
grup3 = [threading.Thread(target=reader, args=(i,)) for i in range(3, 6)]
for t in grup3: t.start()
for t in grup1 + grup2 + grup3: t.join()

for l in log:
    print(l)
print(f"\nNilai akhir: {data['nilai']} (harus 2) | Pelanggaran mutual exclusion: {pelanggaran[0]} (harus 0)")

Reader-0 mulai membaca nilai=0
Reader-1 mulai membaca nilai=0
Reader-2 mulai membaca nilai=0
Writer-1 menulis nilai baru=1
Writer-0 menulis nilai baru=2
Reader-4 mulai membaca nilai=2
Reader-3 mulai membaca nilai=2
Reader-5 mulai membaca nilai=2

Nilai akhir: 2 (harus 2) | Pelanggaran mutual exclusion: 0 (harus 0)


### Uji 2 — Apakah reader baru berisiko starvation jika writer datang terus-menerus?
3 thread writer berulang tanpa henti (selalu ada writer yang menunggu), lalu satu reader baru datang. Reader diberi waktu tunggu 2 detik, kemudian writer dihentikan.

In [4]:
rw = ReadWriteLockPrioritasWriter()
berhenti = threading.Event()
reader_masuk = threading.Event()
jumlah_tulis = [0]

def writer_terus_menerus():
    while not berhenti.is_set():
        rw.mulai_tulis()
        jumlah_tulis[0] += 1
        time.sleep(0.01)
        rw.selesai_tulis()

def satu_reader():
    t0 = time.perf_counter()
    rw.mulai_baca()
    reader_masuk.set()
    print(f"Reader akhirnya mulai membaca setelah {time.perf_counter() - t0:.2f} detik menunggu")
    rw.selesai_baca()

writers = [threading.Thread(target=writer_terus_menerus) for _ in range(3)]
for t in writers: t.start()
time.sleep(0.1)

reader = threading.Thread(target=satu_reader)
reader.start()
reader.join(timeout=2)

if not reader_masuk.is_set():
    print("Reader BELUM mendapat giliran setelah 2 detik -- STARVATION READER TERBUKTI!")
print("Jumlah penulisan selama reader menunggu:", jumlah_tulis[0])

berhenti.set()                       # hentikan writer
for t in writers: t.join()
reader.join(timeout=2)
print("Reader berhasil masuk setelah writer berhenti:", reader_masuk.is_set())

Reader BELUM mendapat giliran setelah 2 detik -- STARVATION READER TERBUKTI!
Jumlah penulisan selama reader menunggu: 207
Reader akhirnya mulai membaca setelah 2.03 detik menunggu
Reader berhasil masuk setelah writer berhenti: True


**Jawaban: Ya, reader baru berisiko starvation.** Selama writer terus berdatangan, `writer_menunggu` tidak pernah bernilai 0, sehingga syarat masuk reader (`writer_menunggu == 0`) tidak pernah terpenuhi. Reader baru baru bisa masuk setelah aliran writer berhenti, seperti terlihat pada Uji 2. Ini kebalikan dari solusi reader-preference, yang berisiko membuat writer starvation.

## Latihan 3 — Sleeping Barber dengan 2 Tukang Cukur

Dua tukang cukur berbagi **satu ruang tunggu** berkapasitas 3 kursi. Ruang tunggu dibuat dengan `queue.Queue(maxsize=3)`:
- Pelanggan memakai `put_nowait()`. Jika antrean penuh (`queue.Full`), pelanggan langsung pergi.
- Tiap tukang cukur memanggil `get()`. Jika antrean kosong, ia tertidur otomatis.
- Pelanggan yang menunggu menunggu `Event` miliknya sendiri, yang di-set oleh tukang cukur setelah selesai mencukur.

In [5]:
import threading
import queue
import time
import random

KURSI_TUNGGU = 3
JUMLAH_TUKANG = 2
JUMLAH_PELANGGAN = 15

ruang_tunggu = queue.Queue(maxsize=KURSI_TUNGGU)   # satu ruang tunggu bersama
log_lock = threading.Lock()
statistik = {"dilayani": [0] * JUMLAH_TUKANG, "pergi": 0}

def catat(msg):
    with log_lock:
        print(msg)

def tukang_cukur(id_tukang):
    while True:
        id_pel, selesai = ruang_tunggu.get()     # tidur jika ruang tunggu kosong
        catat(f"[Tukang-{id_tukang}] mulai mencukur Pelanggan-{id_pel}")
        time.sleep(0.15)
        with log_lock:
            statistik["dilayani"][id_tukang] += 1
        catat(f"[Tukang-{id_tukang}] selesai mencukur Pelanggan-{id_pel}")
        selesai.set()

def pelanggan(id_pel):
    selesai = threading.Event()
    try:
        ruang_tunggu.put_nowait((id_pel, selesai))
        catat(f"[Pelanggan-{id_pel}] masuk ruang tunggu")
    except queue.Full:
        with log_lock:
            statistik["pergi"] += 1
        catat(f"[Pelanggan-{id_pel}] ruang tunggu PENUH, pergi!")
        return
    selesai.wait()                               # tunggu sampai selesai dicukur

for i in range(JUMLAH_TUKANG):
    threading.Thread(target=tukang_cukur, args=(i,), daemon=True).start()

random.seed(1)
threads_pel = []
for i in range(JUMLAH_PELANGGAN):
    time.sleep(random.uniform(0.01, 0.05))
    t = threading.Thread(target=pelanggan, args=(i,))
    t.start()
    threads_pel.append(t)
for t in threads_pel:
    t.join()

print()
for i, n in enumerate(statistik["dilayani"]):
    print(f"Tukang-{i} melayani {n} pelanggan")
print(f"Pelanggan pergi (ruang tunggu penuh): {statistik['pergi']}")
total = sum(statistik["dilayani"]) + statistik["pergi"]
print(f"Total: {total} dari {JUMLAH_PELANGGAN} pelanggan (harus sama)")

[Pelanggan-0] masuk ruang tunggu
[Tukang-0] mulai mencukur Pelanggan-0
[Pelanggan-1] masuk ruang tunggu
[Tukang-1] mulai mencukur Pelanggan-1
[Pelanggan-2] masuk ruang tunggu
[Pelanggan-3] masuk ruang tunggu
[Pelanggan-4] masuk ruang tunggu


[Tukang-0] selesai mencukur Pelanggan-0
[Tukang-0] mulai mencukur Pelanggan-2
[Pelanggan-5] masuk ruang tunggu
[Tukang-1] selesai mencukur Pelanggan-1
[Tukang-1] mulai mencukur Pelanggan-3


[Pelanggan-6] masuk ruang tunggu
[Pelanggan-7] ruang tunggu PENUH, pergi!
[Pelanggan-8] ruang tunggu PENUH, pergi!
[Pelanggan-9] ruang tunggu PENUH, pergi!
[Tukang-0] selesai mencukur Pelanggan-2
[Tukang-0] mulai mencukur Pelanggan-4
[Pelanggan-10] masuk ruang tunggu
[Tukang-1] selesai mencukur Pelanggan-3
[Tukang-1] mulai mencukur Pelanggan-5
[Pelanggan-11] masuk ruang tunggu


[Pelanggan-12] ruang tunggu PENUH, pergi!
[Pelanggan-13] ruang tunggu PENUH, pergi!


[Pelanggan-14] ruang tunggu PENUH, pergi!
[Tukang-0] selesai mencukur Pelanggan-4
[Tukang-0] mulai mencukur Pelanggan-6
[Tukang-1] selesai mencukur Pelanggan-5
[Tukang-1] mulai mencukur Pelanggan-10


[Tukang-0] selesai mencukur Pelanggan-6
[Tukang-0] mulai mencukur Pelanggan-11


[Tukang-1] selesai mencukur Pelanggan-10
[Tukang-0] selesai mencukur Pelanggan-11

Tukang-0 melayani 5 pelanggan
Tukang-1 melayani 4 pelanggan
Pelanggan pergi (ruang tunggu penuh): 6
Total: 15 dari 15 pelanggan (harus sama)


**Cara menentukan tukang cukur berikutnya:** menggunakan satu `Queue` bersama sebagai ruang tunggu. Kedua tukang cukur memanggil `get()` pada antrean yang sama, sehingga tukang cukur yang **lebih dulu bebas** langsung mengambil pelanggan terdepan (FIFO) tanpa perlu penjadwal tambahan. Antrean kosong membuat tukang cukur tidur di `get()`, dan `put_nowait()` yang gagal dengan `queue.Full` menangani penolakan saat 3 kursi penuh.

## Latihan 4 — Refleksi Singkat

Menurut saya yang paling mirip adalah Producer-Consumer. Pola ini hampir selalu muncul pada pengembangan web server. Contohnya aplikasi web yang menerima unggahan foto: request dari pengguna adalah producer yang memasukkan tugas (misalnya kompres gambar atau kirim email) ke antrean, sedangkan beberapa worker adalah consumer yang mengambil tugas tersebut satu per satu. Antrean berbatas mencegah server kehabisan memori saat lonjakan trafik, dan producer tidak perlu menunggu pekerjaan berat selesai. Pola yang sama dipakai pada sistem embedded, saat interrupt sensor (producer) mengisi buffer dan thread pengolah data (consumer) membacanya. Karena itu saya anggap Producer-Consumer paling relevan dengan kebutuhan nyata.